# Original dataset [AI config dataset](https://se-uhd.de/ai-config-dataset/)

In [1]:
import re
import pandas as pd
import numpy as np

In [161]:
# Original dataset
configured_repos = pd.read_csv("../data/configured_repos.csv")
configured_repos = configured_repos.rename(columns={'repo_name': 'repo'})
skills = pd.read_csv("../data/skills.csv")
subagents = pd.read_csv("../data/subagents.csv") # subagent definitions
commands = pd.read_csv("../data/commands.csv") # custom commands
rules = pd.read_csv("../data/rules.csv") # rule files
settings = pd.read_csv("../data/settings.csv") # settings files
hooks = pd.read_csv("../data/hooks.csv") # hook configurations
mcp = pd.read_csv("../data/mcp.csv") # MCP configurations

# Our data
pr_df = pd.read_csv("../data/pull_request_combined.csv")
pr_five_df = pd.read_csv("../data/repo_agent_metrics_comparison_mode_1.csv")
pr_feat_df = pd.read_csv("../data/prs_features.csv")
repo_df = pd.read_csv("../data/repository_combined.csv")
repo_df = repo_df.rename(columns={'id': 'repo_id'})

pr_df = pr_df.merge(
    repo_df[['repo_id', 'full_name']],
    on='repo_id',
    how='left'
).rename(columns={'full_name': 'repo'})

def get_repo_name(url):
    # The pattern defined above
    pattern = r"github\.com[\/:]([^\/\s?#]+?\/[^\/\s?#]+?)(?:\.git)?(?:\/|$)"
    
    match = re.search(pattern, url)
    return match.group(1) if match else None

pr_feat_df['repo'] = pr_feat_df['repo'].map(get_repo_name)

# Add the number of agent-authored PRs to the original repo dataframe
configured_repos = configured_repos.merge(
    pr_feat_df.groupby("repo").size().reset_index(name="agent_prs"),
    on="repo", how='left'
)

# 1. Convert string columns to datetime
# Using utc=True is safer if your timestamps include timezone offsets (like 'Z')
configured_repos['createdAt'] = pd.to_datetime(configured_repos['createdAt'], utc=True)
configured_repos['scanned_at'] = pd.to_datetime(configured_repos['scanned_at'], utc=True)

# 2. Compute the difference (Age)
# This creates a Timedelta column
configured_repos['age_delta'] = configured_repos['scanned_at'] - configured_repos['createdAt']

# 3. Extract specific units
# Age in days (as integer)
configured_repos['repo_age_days'] = configured_repos['age_delta'].dt.days

In [135]:
# pr_commits_df = pd.read_csv("../data/pr_commits_combined.csv")
# pr_commit_details_df = pd.read_csv("../data/pr_commit_details_combined.csv")

# pr_df = pr_df.rename(columns={'id': 'pr_id'})

pr_commit_details_df = pr_commit_details_df.merge(
    pr_df[['pr_id', 'repo', 'number']],
    on='pr_id',
    how='left'
)

In [162]:
def generate_repo_stats_table(df: pd.DataFrame):
    """
    Generate a LaTeX table with repository statistics from GitHub PR data.
'', ''
    Args:
        df (pd.DataFrame): DataFrame containing GitHub PR data with repository metadata
        output_file (str): Path to output LaTeX file
    """
    # Step 1: Remove duplicate repositories (keep first occurrence)
    df_unique = df.sort_values("stargazers", ascending=0).drop_duplicates(
        subset=['repo'], keep='first').copy()

    # Count security PRs per repository
    # agentic_prs = df.groupby('repo').size()
    # df_unique['agentic_pr_count'] = df_unique['repo'].map(agentic_prs).fillna(0)

    # Step 3: Prepare the statistics table
    metrics = {
        'Age (days)': 'repo_age_days',
        'Stars': 'stargazers',
        'Commits': 'commits',
        'Agentic-PRs': 'agent_prs',
        'Total PRs': 'totalPullRequests',
        'Contributors': 'contributors',
        # 'Users': 'mentionableUsersCount'
    }

    # Calculate statistics for each metric
    stats_data = []
    for metric_name, col_name in metrics.items():
        values = df_unique[col_name]
        stats_data.append({
            'Metric': metric_name,
            'Min.': np.min(values),
            'Median': np.median(values),
            'Mean': np.mean(values),
            'Max.': np.max(values)
        })

    # Create stats DataFrame
    stats_df = pd.DataFrame(stats_data)
    stats_df['Max.'] = stats_df['Max.'].apply(lambda x: f"{x:,}")
    stats_df['Median'] = stats_df['Median'].apply(lambda x: f"{x:,.2f}")
    stats_df['Mean'] = stats_df['Mean'].apply(lambda x: f"{x:,.2f}")

    # Step 4: Generate LaTeX table
    latex_table = stats_df.to_latex(
        index=False,
        column_format='lrrrr',
        float_format="{:0.2f}".format,
        caption=f'Statistics of the {df['repo'].nunique():,} studied projects.',
        label='tab:repo-stats',
        position='t!'
    )
    latex_table = latex_table.replace(".00", "")
    # Save to file
    # with open(output_file, 'w') as f:
    #     f.write(latex_table)

    return latex_table, df_unique

#### Stats of the project with at least 5 agent-authored and 5 after the creation of the first instruction file

In [4]:
COUNT_PR_THRESHOLD = 5

In [6]:
five_pr_mask = (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD)&(pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD)
repos_with_5_pr_bef_aft = pr_five_df.loc[
    five_pr_mask,
    'repo'
].unique()

In [163]:
latex_output, repo_stat_df = generate_repo_stats_table(configured_repos[configured_repos['repo'].isin(repos_with_5_pr_bef_aft)])

In [164]:
print(latex_output)

\begin{table}[t!]
\caption{Statistics of the 167 studied projects.}
\label{tab:repo-stats}
\begin{tabular}{lrrrr}
\toprule
Metric & Min. & Median & Mean & Max. \\
\midrule
Age (days) & 571 & 2,021 & 2,278.29 & 6,357.0 \\
Stars & 10 & 1,692 & 9,002.95 & 145,711.0 \\
Commits & 375 & 4,033 & 8,675.92 & 142,284.0 \\
Agentic-PRs & NaN & nan & nan & nan \\
Total PRs & 45 & 2,226 & 5,236.34 & 73,940.0 \\
Contributors & 5 & 83 & 135.26 & 463.0 \\
\bottomrule
\end{tabular}
\end{table}



#### Stats for projects with at least X agent-authored PR after the creation of context files

In [165]:
# Number of configured projects
num_configured_pro = configured_repos['repo'].unique()
num_pro_with_prs = pr_df['repo'].unique()
num_agentic_prs = pr_feat_df['repo'].isin(repos_with_5_pr_bef_aft).sum()

# Number of projects that have at least one agent-authored PRs (up to 2026-03-03)
print("Percentage of projects that have at least one agent-authored PR:", round(100 * (len(num_pro_with_prs) / len(num_configured_pro)), 2), f"({len(num_pro_with_prs)} out of {len(num_configured_pro)})")
print(f"The number of agent-authored PRs associated with the 167 projects: {num_agentic_prs}")
print(f"The distribution of PRs for the {len(num_pro_with_prs)} with at least one agent-authored PR:")

num_proj_with_five_prs_bef_aft = pr_five_df.loc[
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD)&
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have skills?
num_proj_with_skills = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(skills['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have sub-agents
num_proj_with_subagents = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(subagents['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have commands
num_proj_with_commands = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(commands['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have hooks
num_proj_with_hooks = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(hooks['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have mcp
num_proj_with_mcp = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(mcp['repo_name'].values)),
    'repo'
].unique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have settings
num_proj_with_settings = pr_five_df.loc[
    (pr_five_df['pr_count_after']>=COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(settings['repo_name'].values)),
    'repo'
].unique()

print(f"Percentage of projects that have at least {COUNT_PR_THRESHOLD} agent-authored PRs before and {COUNT_PR_THRESHOLD} after the creation of context files:", round(100 * (len(num_proj_with_five_prs_bef_aft) / len(num_configured_pro)), 2), f"({len(num_proj_with_five_prs_bef_aft)} out of {len(num_configured_pro)})")
print("The number of project with at least five PRs before and five after the creation of context files:")
print(f"{' '*8} 1. Skills: {round(100 * len(num_proj_with_skills) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 2. Sub-agents: {round(100 * len(num_proj_with_subagents) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 3. Commands: {round(100 * len(num_proj_with_commands) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 4. Hooks: {round(100 * len(num_proj_with_hooks) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 5. Settings: {round(100 * len(num_proj_with_settings) / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 6. MCP: {round(100 * len(num_proj_with_mcp) / len(num_proj_with_five_prs_bef_aft), 2)}")

Percentage of projects that have at least one agent-authored PR: 47.66 (2258 out of 4738)
The number of agent-authored PRs associated with the 167 projects: 0
The distribution of PRs for the 2258 with at least one agent-authored PR:
Percentage of projects that have at least 5 agent-authored PRs before and 5 after the creation of context files: 17.88 (847 out of 4738)
The number of project with at least five PRs before and five after the creation of context files:
         1. Skills: 19.72
         2. Sub-agents: 12.16
         3. Commands: 11.57
         4. Hooks: 5.31
         5. Settings: 16.88
         6. MCP: 4.13


#### Stats for projects with at least X agent-authored PR after the creation of context files

In [166]:
# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have skills?
num_proj_with_skills = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(skills['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have sub-agents
num_proj_with_subagents = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(subagents['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have commands
num_proj_with_commands = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(commands['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have hooks
num_proj_with_hooks = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(hooks['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have mcp
num_proj_with_mcp = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(mcp['repo_name'].values)),
    'repo'
].nunique()

# How many projects with at least COUNT_PR_THRESHOLD agent-authored PRs and COUNT_PR_THRESHOLD after have settings
num_proj_with_settings = pr_five_df.loc[
    (pr_five_df['pr_count_after']<COUNT_PR_THRESHOLD) &
    # (pr_five_df['pr_count_before']>=COUNT_PR_THRESHOLD) &
    (pr_five_df['repo'].isin(settings['repo_name'].values)),
    'repo'
].nunique()

print(f"The number of project with at least {COUNT_PR_THRESHOLD} after the creation of context files:")
print(f"{' '*8} 1. Skills: {round(100 * num_proj_with_skills / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 2. Sub-agents: {round(100 * num_proj_with_subagents / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 3. Commands: {round(100 * num_proj_with_commands / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 4. Hooks: {round(100 * num_proj_with_hooks / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 5. Settings: {round(100 * num_proj_with_settings / len(num_proj_with_five_prs_bef_aft), 2)}")
print(f"{' '*8} 6. MCP: {round(100 * num_proj_with_mcp / len(num_proj_with_five_prs_bef_aft), 2)}")

The number of project with at least 5 after the creation of context files:
         1. Skills: 12.28
         2. Sub-agents: 5.55
         3. Commands: 5.31
         4. Hooks: 2.13
         5. Settings: 7.79
         6. MCP: 3.31


#### The comparison among projects have have skills vs. those that do not

##### Skills

In [31]:
repos_with_skills = skills['repo_name'].unique()

skills_df = pr_five_df[
    five_pr_mask &
    pr_five_df['repo'].isin(repos_with_skills)
]

non_skills_df = pr_five_df[
    five_pr_mask &
    (~pr_five_df['repo'].isin(repos_with_skills))
]

# print("SKILLS stats among projects that have skills vs. those that do not:")
# for metric in ['closure_time', 'num_files', 'code_churn', 'num_commits', 'comments', 'description_length', 'num_inline_comments']:
#     skill_val = skills_df['metric'].median()
#     non_skill_val = non_skills_df[metric].median()
    
#     print(f"Metric: {metric}, with skills: {skill_val}, without skills: {non_skill_val}")
non_skills_df['merge_impro_pct'].describe()

/home/as98450/projects/context-files-generator/.venv/lib/python3.12/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


count    138.0000
mean          inf
std           NaN
min     -100.0000
25%      -15.4525
50%        0.0000
75%       24.5050
max           inf
Name: merge_impro_pct, dtype: float64

##### Subagents

In [170]:
repos_with_subagents = subagents['repo_name'].unique()

subagent_df = pr_five_df[
    five_pr_mask &
    pr_five_df['repo'].isin(repos_with_subagents)
]

non_subagent_df = pr_five_df[
    five_pr_mask &
    (~pr_five_df['repo'].isin(repos_with_subagents))
]

In [ ]:
subagent_df['merge_impro_pct'].describe()

##### Commands

In [48]:
repos_with_commands = rules['repo_name'].unique()

command_df = pr_five_df[
    five_pr_mask &
    pr_five_df['repo'].isin(repos_with_commands)
]

non_command_df = pr_five_df[
    five_pr_mask &
    (~pr_five_df['repo'].isin(repos_with_commands))
]

In [67]:
pr_five_df[five_pr_mask].sort_values("merge_impro_pct")[:10][['repo', 'merge_impro_pct', 'pr_count_before', 'pr_count_after']]

,repo,merge_impro_pct,pr_count_before,pr_count_after
311,kubernetes-sigs/headlamp,-100.00,12.0,5.0
596,danny-avila/LibreChat,-88.57,16.0,20.0
1072,stellar/stellar-cli,-76.19,5.0,7.0
726,envoyproxy/toolshed,-59.90,40.0,171.0
520,semaphoreui/semaphore,-51.92,5.0,26.0
187,apache/datafusion,-51.32,40.0,17.0
597,valkey-io/valkey-glide,-48.15,56.0,12.0
1111,lmnr-ai/lmnr,-47.39,11.0,115.0
1623,microsoft/go-sqlcmd,-46.43,6.0,14.0
928,tetherless-world/whyis,-41.67,6.0,12.0


In [128]:
pr_df[pr_df['pr_id']==3825583836]

,pr_id,number,title,body,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url,repo
1197,3825583836,123313,Fix Base64.DecodeFromUtf8 consuming whitespace...,## Description\n\n`Base64.DecodeFromUtf8()` in...,198982749,Copilot,closed,2026-01-17T17:49:28Z,2026-01-26T13:20:19Z,2026-01-26T13:20:19Z,210716005,https://github.com/dotnet/runtime,https://github.com/dotnet/runtime/pull/123313,dotnet/runtime


In [144]:
pr_commit_details_df[pr_commit_details_df['filename'].str.contains("package.json")&(pr_commit_details_df['repo'].isin(repos_with_5_pr_bef_aft))]

,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,additions,deletions,changes,patch,repo,number
152971,1c55383192392070d4dba445de03d016f892ab1a,3683296801,agocke,web-flow,Merge branch 'main' into copilot/switch-ci-foc...,5055,2869,2186,src/mono/sample/wasm/console-node/wwwroot/pack...,renamed,0,0,0,NaN,dotnet/runtime,122085
165991,f2136dbe29f8a0c15ef2871ea31e5dc4cf2b5f13,3895614764,stephentoub,web-flow,Merge branch 'main' into copilot/fix-ziparchiv...,25842,17023,8819,src/mono/sample/wasm/console-node/wwwroot/pack...,modified,2,2,4,NaN,dotnet/runtime,123991
166313,4fa51fb2bb55939f19c787e62dcf5ad41dd1939d,3895592793,stephentoub,web-flow,Merge branch 'main' into copilot/sync-extended...,25842,17023,8819,src/mono/sample/wasm/console-node/wwwroot/pack...,modified,2,2,4,NaN,dotnet/runtime,123990
179194,e3f1d007b4d4491ae05eaad4cccadd84670be261,3825583836,MihaZupan,web-flow,Merge branch 'main' into copilot/fix-base64-de...,6133,4770,1363,src/native/package.json,modified,2,2,4,"@@ -12,8 +12,8 @@\n ""scripts"": {\n ""roll...",dotnet/runtime,123313
179692,7902812883c78719da01206cc7ce84a9799441f0,3824848500,maraf,web-flow,Merge branch 'main' into copilot/refactor-weba...,4977,3823,1154,src/native/package.json,modified,2,2,4,"@@ -12,8 +12,8 @@\n ""scripts"": {\n ""roll...",dotnet/runtime,123304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3349984,3bbe696e8d46fdf4564635834f23a0e945b1327d,3963777623,Copilot,Copilot,Resolve merge conflict: incorporate tar overri...,9,5,4,package.json,modified,2,1,3,"@@ -11,6 +11,7 @@\n ""private"": true,\n ""ov...",openai/openai-dotnet,962
3349986,0585ec25ffe1feaffb2a90f97cb280a64715f065,3963777623,Copilot,Copilot,Fix merge conflict: isolate ajv change to pack...,7,3,4,package.json,modified,0,1,1,"@@ -10,7 +10,6 @@\n },\n ""private"": true,\...",openai/openai-dotnet,962
3349987,fb46b03dd6f814d6206078888a9bf427017c36b6,3963777623,jorgerangel-msft,jorgerangel-msft,fix merge conflict,3,2,1,package.json,modified,2,1,3,"@@ -11,6 +11,7 @@\n ""private"": true,\n ""ov...",openai/openai-dotnet,962
3349990,63f86daa915ba8cb65747febc3598600ec0b14d7,3963772350,Copilot,Copilot,fix: upgrade transitive tar dependency to >=7....,282,213,69,package.json,modified,2,1,3,"@@ -10,6 +10,7 @@\n },\n ""private"": true,\...",openai/openai-dotnet,960


#### 

In [171]:
first_ai_gen_df = pr_five_df[
    five_pr_mask &
    (pr_five_df['first_commit_context_ai_created'] == True)
    # pr_five_df['repo'].isin(repos_with_subagents)
]

non_first_ai_gen_df = pr_five_df[
    five_pr_mask &
    (pr_five_df['first_commit_context_ai_created'] == False)
    # (~pr_five_df['repo'].isin(repos_with_subagents))
]

In [178]:
first_ai_gen_df['word_count'].describe(), non_first_ai_gen_df['word_count'].describe()

(count       44.000000
 mean      1871.818182
 std       2256.855007
 min        178.000000
 25%        806.750000
 50%       1043.500000
 75%       1918.250000
 max      13143.000000
 Name: word_count, dtype: float64,
 count      123.000000
 mean      1811.398374
 std       2647.411934
 min         62.000000
 25%        519.500000
 50%        972.000000
 75%       1814.000000
 max      21437.000000
 Name: word_count, dtype: float64)